# Step 04 — Common Initial Fresh-FV Share (2007)

Calculate **one pooled 2007 fresh-FV dollar share** for households in the current HMM sample:

\[
\pi_{\mathrm{FV},2007}=\frac{\sum_i\mathrm{FreshFVSpend}_{i,2007}}{\sum_i\mathrm{FoodSpend}_{i,2007}}.
\]

This is **not** the unweighted average of household shares. It supplies a common numerical **starting value** for the initial HMM healthy-state probability, not a fixed estimated parameter.

The 2007 produce and Magnet extraction TSVs are already prepared. All identifiers remain strings; regular and Magnet spending are reconciled before summation. Output is a **single-row Parquet with one column** (`initial_fresh_fv_share_2007`).


In [ ]:
from pathlib import Path
import pandas as pd
import duckdb

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
REPO_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_cycle")
HMS_ROOT = Path(r"C:/Users/rwen13/Documents/2004-2020_Panel_Data/nielsen_extracts/HMS")

PRODUCE_FILE = DATA_ROOT / "intermediate/produce_purchases/fv_purchases_2007.tsv"
MAGNET_FILE = DATA_ROOT / "intermediate/magnet_food_purchases/magnet_food_purchases_2007.tsv"
PURCHASES_FILE = HMS_ROOT / "2007/Annual_Files/purchases_2007.tsv"
TRIPS_FILE = HMS_ROOT / "2007/Annual_Files/trips_2007.tsv"
PRODUCTS_FILE = HMS_ROOT / "Master_Files/Latest/products.tsv"
FOOD_LIST_FILE = REPO_ROOT / "data_cleaning/upc_classification/food_list.xlsx"
ANNUAL_FILE = DATA_ROOT / "processed/hmm_panel/hmm_master/hmm_annual_covariates_income_imputed.parquet"
OUTPUT_DIR = DATA_ROOT / "processed/hmm_panel/hmm_master"
AUDIT_DIR = OUTPUT_DIR / "validation"
OUTPUT_FILE = OUTPUT_DIR / "hmm_initial_fresh_fv_share_2007.parquet"

for name, path in {
    "Produce": PRODUCE_FILE, "Magnet": MAGNET_FILE,
    "Raw purchases": PURCHASES_FILE, "Raw trips": TRIPS_FILE,
    "Product master": PRODUCTS_FILE, "Food module list": FOOD_LIST_FILE,
    "Current HMM annual panel": ANNUAL_FILE,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print(f"Found {name}: {path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "temp").mkdir(parents=True, exist_ok=True)

def sql_path(path):
    return "'" + path.as_posix().replace("'", "''") + "'"

def source_tsv(path):
    # All columns begin as text; household/trip/UPC join keys cannot lose zeros.
    return f"read_csv({sql_path(path)}, delim='\t', header=true, all_varchar=true)"

con = duckdb.connect(str(DATA_ROOT / "temp/hpc_step04_2007.duckdb"))
con.execute("SET memory_limit='4GB'")
con.execute(f"SET temp_directory={sql_path(DATA_ROOT / 'temp')}")


In [ ]:
# The food-module lookup is a small permitted rule file, not Nielsen purchase data.
# Normalize MODULE codes only; never normalize household, UPC, or trip identifiers.
food = pd.read_excel(FOOD_LIST_FILE, dtype={"product_module_code": "string"})
if "product_module_code" not in food.columns:
    raise ValueError("food_list.xlsx must contain product_module_code")
food_modules = (
    food["product_module_code"].dropna().str.strip()
    .str.replace(r"\.0+$", "", regex=True)
    .str.lstrip("0").replace("", "0").drop_duplicates()
)
if food_modules.empty:
    raise ValueError("The food-module list is empty")
con.register("food_module_frame", pd.DataFrame({"module_code": food_modules.astype(str)}))
con.execute("CREATE OR REPLACE TEMP TABLE food_modules AS SELECT DISTINCT module_code FROM food_module_frame")

# Restrict processing to households retained in the HMM model sample.
con.execute(f"""
CREATE OR REPLACE TEMP TABLE hmm_households AS
SELECT DISTINCT CAST(hhid AS VARCHAR) AS hhid
FROM read_parquet({sql_path(ANNUAL_FILE)})
WHERE hhid IS NOT NULL
""")
print("HMM households:", con.execute("SELECT COUNT(*) FROM hmm_households").fetchone()[0])


In [ ]:
# Read extracted regular produce and Magnet food from 2007.
# Use 2007 calendar purchase dates rather than trusting file-year labels.
con.execute(f"""
CREATE OR REPLACE VIEW produce_2007 AS
SELECT CAST(household_code AS VARCHAR) AS hhid,
       CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       TRY_CAST(final_price_paid AS DOUBLE) AS net_amount
FROM {source_tsv(PRODUCE_FILE)}
WHERE YEAR(TRY_CAST(purchase_date AS DATE)) = 2007
  AND TRY_CAST(fv_class AS INTEGER) = 1
""")
con.execute(f"""
CREATE OR REPLACE VIEW magnet_2007 AS
SELECT CAST(household_code AS VARCHAR) AS hhid,
       CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       TRY_CAST(magnet_food_spend_increment_capped AS DOUBLE) AS food_amount,
       TRY_CAST(magnet_fresh_fv_spend_increment_capped AS DOUBLE) AS fresh_amount
FROM {source_tsv(MAGNET_FILE)}
WHERE YEAR(TRY_CAST(purchase_date AS DATE)) = 2007
  AND TRY_CAST(is_magnet_food AS INTEGER) = 1
""")

# Preserve repeated purchase lines; create a key-level membership set only for
# anti-joining raw Regular purchases, not for deleting or summing Magnet lines.
con.execute("""
CREATE OR REPLACE TEMP TABLE magnet_keys AS
SELECT trip_code_uc, upc, upc_ver_uc, COUNT(*) AS magnet_line_count
FROM magnet_2007 GROUP BY 1,2,3
""")

# Produce and Magnet should not double-count the same purchase UPC/trip.
produce_magnet_overlap = con.execute("""
SELECT COUNT(*) FROM produce_2007 p
JOIN magnet_keys m USING (trip_code_uc, upc, upc_ver_uc)
""").fetchone()[0]
if produce_magnet_overlap:
    raise ValueError(f"Produce/Magnet overlap: {produce_magnet_overlap} lines. Reconcile before calculating shares.")
print("Regular Fresh FV vs Magnet overlapping lines:", produce_magnet_overlap)


In [ ]:
# Source purchases are read only for the 2007 source file.
# Join raw purchases to actual trip dates and the product master using string keys.
# The product reference must have a unique (upc, upc_ver_uc) key.
con.execute(f"""
CREATE OR REPLACE VIEW raw_trips AS
SELECT CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(household_code AS VARCHAR) AS hhid,
       TRY_CAST(purchase_date AS DATE) AS purchase_date
FROM {source_tsv(TRIPS_FILE)}
""")
con.execute(f"""
CREATE OR REPLACE VIEW product_modules AS
SELECT CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       COALESCE(NULLIF(LTRIM(REGEXP_REPLACE(TRIM(product_module_code), '\.0+$', ''), '0'), ''), '0') AS module_code
FROM {source_tsv(PRODUCTS_FILE)}
""")
product_duplicates = con.execute("""
SELECT COUNT(*) FROM (
  SELECT upc, upc_ver_uc FROM product_modules
  GROUP BY 1,2 HAVING COUNT(*) > 1
)
""").fetchone()[0]
if product_duplicates:
    raise ValueError(f"Duplicate UPC/version pairs in product master: {product_duplicates}")
trip_duplicates = con.execute("""
SELECT COUNT(*) FROM (
  SELECT trip_code_uc FROM raw_trips GROUP BY 1 HAVING COUNT(*) > 1
)
""").fetchone()[0]
if trip_duplicates:
    raise ValueError(f"Duplicate trip_code_uc in 2007 trips: {trip_duplicates}")

con.execute(f"""
CREATE OR REPLACE VIEW raw_2007 AS
SELECT t.hhid, p.trip_code_uc, p.upc, p.upc_ver_uc,
       TRY_CAST(p.total_price_paid AS DOUBLE) AS total_price_paid,
       TRY_CAST(p.coupon_value AS DOUBLE) AS coupon_value
FROM {source_tsv(PURCHASES_FILE)} p
JOIN raw_trips t ON p.trip_code_uc = t.trip_code_uc
WHERE YEAR(t.purchase_date) = 2007
""")

# Reconcile Magnet multiplicities against source purchases.
# Repeated UPCs within a trip are valid purchase lines, NOT duplicates to delete.
con.execute("""
CREATE OR REPLACE TEMP TABLE source_magnet_counts AS
SELECT r.trip_code_uc, r.upc, r.upc_ver_uc, COUNT(*) AS raw_line_count
FROM raw_2007 r
JOIN magnet_keys m USING (trip_code_uc, upc, upc_ver_uc)
GROUP BY 1,2,3
""")
key_mismatches = con.execute("""
SELECT COUNT(*) FROM magnet_keys m
LEFT JOIN source_magnet_counts r USING (trip_code_uc, upc, upc_ver_uc)
WHERE r.raw_line_count IS NULL OR r.raw_line_count <> m.magnet_line_count
""").fetchone()[0]
if key_mismatches:
    raise ValueError(
        f"Magnet/raw line multiplicities differ for {key_mismatches} UPC-trip keys. "
        "Check source-year overlap or purchase-date alignment; do not silently deduplicate."
    )
print("Magnet/raw UPC-trip count mismatches:", key_mismatches)


In [ ]:
# Sum annual spending within the household. The Magnet overlap is removed
# from Regular food BEFORE aggregation. All dollars use max(price-coupon, 0).
# Missing prices are not treated as observed zeros; report their number below.
con.execute("""
CREATE OR REPLACE TEMP TABLE regular_food_by_hh AS
SELECT r.hhid,
       SUM(GREATEST(r.total_price_paid - COALESCE(r.coupon_value, 0), 0)) AS regular_food_spend,
       COUNT(*) AS regular_food_lines,
       COUNT(*) FILTER (WHERE r.total_price_paid IS NULL) AS regular_missing_price_lines
FROM raw_2007 r
JOIN product_modules p ON r.upc=p.upc AND r.upc_ver_uc=p.upc_ver_uc
JOIN food_modules f ON p.module_code=f.module_code
ANTI JOIN magnet_keys m ON r.trip_code_uc=m.trip_code_uc AND r.upc=m.upc AND r.upc_ver_uc=m.upc_ver_uc
GROUP BY 1
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE regular_fresh_by_hh AS
SELECT hhid,
       SUM(GREATEST(net_amount, 0)) AS regular_fresh_fv_spend,
       COUNT(*) AS fresh_lines,
       COUNT(*) FILTER (WHERE net_amount IS NULL) AS fresh_missing_price_lines
FROM produce_2007 GROUP BY 1
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE magnet_by_hh AS
SELECT hhid,
       SUM(food_amount) AS magnet_food_spend,
       SUM(fresh_amount) AS magnet_fresh_fv_spend,
       COUNT(*) AS magnet_lines,
       COUNT(*) FILTER (WHERE food_amount IS NULL OR fresh_amount IS NULL) AS magnet_missing_price_lines
FROM magnet_2007 GROUP BY 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE household_spending AS
SELECT h.hhid,
       COALESCE(r.regular_food_spend,0) AS regular_food_spending_2007,
       COALESCE(m.magnet_food_spend,0) AS magnet_food_spending_2007,
       COALESCE(f.regular_fresh_fv_spend,0) AS regular_fresh_fv_spending_2007,
       COALESCE(m.magnet_fresh_fv_spend,0) AS magnet_fresh_fv_spending_2007,
       COALESCE(r.regular_food_spend,0)+COALESCE(m.magnet_food_spend,0) AS total_food_spending_2007,
       COALESCE(f.regular_fresh_fv_spend,0)+COALESCE(m.magnet_fresh_fv_spend,0) AS fresh_fv_spending_2007,
       COALESCE(r.regular_food_lines,0) AS regular_food_lines,
       COALESCE(f.fresh_lines,0) AS regular_fresh_lines,
       COALESCE(m.magnet_lines,0) AS magnet_lines,
       COALESCE(r.regular_missing_price_lines,0) + COALESCE(f.fresh_missing_price_lines,0) + COALESCE(m.magnet_missing_price_lines,0) AS missing_price_lines
FROM hmm_households h
LEFT JOIN regular_food_by_hh r USING(hhid)
LEFT JOIN regular_fresh_by_hh f USING(hhid)
LEFT JOIN magnet_by_hh m USING(hhid)
""")

# No 2007 spending or a zero denominator means undefined share (NULL), not 0.
# Fresh FV is a subset of all food purchases; violations should block output.
invalid = con.execute("""
SELECT COUNT(*) FROM household_spending
WHERE fresh_fv_spending_2007 > total_food_spending_2007 + 0.01
   OR fresh_fv_spending_2007 < -0.01 OR total_food_spending_2007 < -0.01
""").fetchone()[0]
if invalid:
    raise ValueError(f"{invalid} households have fresh FV spending greater than food spending or negative totals")



In [ ]:
# One pooled dollar share across all current HMM households (not a mean of household shares).
# All components below are already net of coupons, capped at zero, and adjusted
# for Regular/Magnet overlap in the preceding cells.
result = con.execute("""
SELECT
    SUM(fresh_fv_spending_2007) AS fresh_fv_dollars,
    SUM(total_food_spending_2007) AS food_dollars,
    COUNT(*) AS hmm_households,
    COUNT(*) FILTER (WHERE total_food_spending_2007 > 0) AS households_with_food_spend,
    SUM(missing_price_lines) AS missing_price_lines
FROM household_spending
""").fetchone()

fresh_fv_dollars, food_dollars, n_households, n_food_households, missing_price_lines = result
if food_dollars is None or food_dollars <= 0:
    raise ValueError("Cannot calculate common initial share: total 2007 food spending must be positive")
common_share = float(fresh_fv_dollars / food_dollars)
if not (0 <= common_share <= 1):
    raise ValueError(f"Initial share is outside [0, 1]: {common_share}")

# Write exactly ONE numeric value, with no household identifiers or household-level rows.
con.execute(f"""
COPY (
    SELECT CAST({common_share:.17g} AS DOUBLE) AS initial_fresh_fv_share_2007
) TO {sql_path(OUTPUT_FILE)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f"initial_fresh_fv_share_2007 = {common_share:.8f}")
print(f"HMM households: {n_households:,}; households with positive 2007 food spending: {n_food_households:,}")
print(f"2007 Fresh FV dollars: {fresh_fv_dollars:,.2f}; Food dollars: {food_dollars:,.2f}")
print(f"Purchase lines with missing prices (audit): {missing_price_lines:,}")
print(f"Saved single-value Parquet: {OUTPUT_FILE}")
con.close()


### Interpretation

- The file contains **one row and one numeric column**. It is not a household-level dataset.
- The pooled dollar share is calculated from total fresh-FV and food spending across the selected HMM household sample.
- The result may be used as a starting value for a common HMM initial-state probability; it is **not** itself a latent-state probability estimate.
- This notebook uses purchase lines from the **2007 source-year file**; source-year spillover must be checked separately if needed.
